# Layer 1 — Dynamic Factor Model Backbone (Final merged implementation)

This notebook is the **definitive merged Layer 1 implementation** for:

> **Real-Time GDP Growth Nowcasting using a Hybrid Dynamic Factor Model with Machine Learning Residual Correction**

What this notebook does:

- audits the **actual local repository** and maps the real data files before modeling;
- compares the **four numbered candidate notebooks** and **four numbered helper modules** that exist in the repository;
- preserves the repository’s **first-day-of-month / first-day-of-quarter** timestamp semantics;
- constructs a **vintage-aware GDP target history** and separate **truth tables** from the real workbooks;
- ingests and transforms the **monthly FRED-MD vintage files** from the repository;
- aligns monthly predictors with quarterly GDP in a **mixed-frequency state-space DFM**;
- estimates the DFM via **EM + Kalman filter / smoother**;
- produces **GDP nowcasts**, **news decomposition**, diagnostics, and **Layer 2 artifacts**.

This final version merges the strongest structural, diagnostic, and mixed-frequency modeling ideas from the four candidate implementations while removing duplicated or conflicting logic.



## Non-negotiable date rule

The repository does **not** use artificial month-end timestamps as its native convention.

This notebook therefore:

- treats strings such as `2026-01-01` as **January 2026**, not as a month-end date;
- treats quarter markers using **period semantics** and sequence-aware parsing;
- avoids invalid synthetic dates such as `2026-02-29`;
- only converts to timestamp form when a downstream library requires it, and then uses the **period start timestamp** rather than a fabricated month-end.

Internally, audit and alignment logic use `pandas.Period` objects wherever possible.


In [ ]:
from __future__ import annotations

import ast
import importlib
import json
import hashlib
import platform
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

try:
    import dfm_layer1_utils_final as _layer1_utils
    _layer1_utils = importlib.reload(_layer1_utils)

    from dfm_layer1_utils_final import (
        ProtocolConfig,
        _assert_news_impact_date_supported,
        _coerce_impact_date_for_model_index,
        apply_tcodes_to_snapshot,
        as_model_index,
        build_factor_mapping,
        build_monotonic_mixed_frequency_endog,
        build_quarterly_target_series_for_vintage,
        build_repo_catalog,
        build_target_and_truth_objects,
        choose_canonical_md_manifest,
        choose_canonical_qd_manifest,
        choose_vintage_schedule,
        completion_checklist_frame,
        compute_block_coverage,
        extract_nowcast_from_results,
        fit_dfm_single_vintage,
        get_quarter_end_month,
        infer_period_frequency_from_values,
        inspect_csv_schema,
        inspect_excel_workbook,
        load_fred_snapshot,
        locate_repo_root,
        make_diagnostics_row,
        model_index_audit_frame,
        model_prediction_index_audit,
        oriented_factor_states,
        parse_periodish,
        protocol_summary_frame,
        run_layer1_dfm,
        select_monthly_panel,
        select_target_workbooks,
        serialize_protocol,
        stable_subset_metadata,
        summarize_manifest,
        within_quarter_origin,
        quarter_of_vintage,
    )
except ImportError as exc:
    raise ImportError(
        "dfm_layer1_utils_final.py was not found next to the notebook. "
        "Place the final helper file in the repository root (or the notebook working directory) and rerun."
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:

import statsmodels
import openpyxl

env = pd.DataFrame(
    [
        {"component": "python", "version": platform.python_version()},
        {"component": "pandas", "version": pd.__version__},
        {"component": "numpy", "version": np.__version__},
        {"component": "statsmodels", "version": statsmodels.__version__},
        {"component": "openpyxl", "version": openpyxl.__version__},
    ]
)
display(env)


## 1. Project setup

In [ ]:

# If the notebook is placed inside the repository (recommended), auto-detection should work.
# Otherwise, set REPO_ROOT_HINT to the absolute local path of the cloned repository.
REPO_ROOT_HINT = None

REPO_ROOT = Path(REPO_ROOT_HINT) if REPO_ROOT_HINT else locate_repo_root(Path.cwd())
if REPO_ROOT is None or not REPO_ROOT.exists():
    raise FileNotFoundError(
        "Could not locate the Nowcasting-GDP-Growth repository automatically. "
        "Set REPO_ROOT_HINT to the local repository path and rerun."
    )

RAW_DIR = REPO_ROOT / "data" / "raw"
OUTPUT_DIR = REPO_ROOT / "outputs" / "layer1_dfm_final"

print("Repository root:", REPO_ROOT)
print("Raw data directory:", RAW_DIR)
print("Output directory:", OUTPUT_DIR)


## 2. Repository audit and file mapping

In [ ]:

catalog = build_repo_catalog(REPO_ROOT)

catalog_summary = (
    catalog.groupby("group", dropna=False)
    .agg(
        n_files=("path", "size"),
        first_vintage=("vintage_period", "min"),
        last_vintage=("vintage_period", "max"),
        total_size_mb=("size_mb", "sum"),
    )
    .reset_index()
    .sort_values(["group"])
)

md_manifest = choose_canonical_md_manifest(catalog)
qd_manifest = choose_canonical_qd_manifest(catalog)
target_workbooks = select_target_workbooks(REPO_ROOT)

manifest_summary = pd.concat(
    [
        summarize_manifest(md_manifest, "Canonical monthly predictor manifest"),
        summarize_manifest(qd_manifest, "Canonical quarterly predictor manifest"),
    ],
    ignore_index=True,
)

workbook_map = pd.DataFrame(
    [
        {"object_name": k, "path": str(v.relative_to(REPO_ROOT))}
        for k, v in sorted(target_workbooks.items())
    ]
)

display(catalog_summary)
display(manifest_summary)
display(workbook_map)


In [ ]:

def hash_file(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

hash_checks = []

md_current = REPO_ROOT / "data" / "raw" / "FRED-MD-MONTHLY" / "current.csv"
if md_current.exists() and not md_manifest.empty:
    latest_md = REPO_ROOT / md_manifest.iloc[-1]["path"]
    hash_checks.append(
        {
            "file_group": "monthly current vs latest dated file",
            "current_path": str(md_current.relative_to(REPO_ROOT)),
            "latest_dated_path": str(latest_md.relative_to(REPO_ROOT)),
            "same_bytes": hash_file(md_current) == hash_file(latest_md),
        }
    )

qd_current = REPO_ROOT / "data" / "raw" / "FRED-QD-QUARTERLY" / "current.csv"
if qd_current.exists() and not qd_manifest.empty:
    latest_qd = REPO_ROOT / qd_manifest.iloc[-1]["path"]
    hash_checks.append(
        {
            "file_group": "quarterly current vs latest dated file",
            "current_path": str(qd_current.relative_to(REPO_ROOT)),
            "latest_dated_path": str(latest_qd.relative_to(REPO_ROOT)),
            "same_bytes": hash_file(qd_current) == hash_file(latest_qd),
        }
    )

display(pd.DataFrame(hash_checks))


### Raw file inspection and schema understanding


In [ ]:

def compact_csv_inspection(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    info = inspect_csv_schema(path)
    summary = pd.DataFrame(
        [
            {
                "path": str(Path(info["path"]).relative_to(REPO_ROOT)),
                "date_col": info["date_col"],
                "inferred_freq": info["inferred_freq"],
                "metadata_row_count": info["metadata_row_count"],
                "has_tcode_row": info["has_tcode_row"],
                "n_rows": info["n_rows"],
                "n_cols": info["n_cols"],
                "min_timestamp": info["min_timestamp"],
                "max_timestamp": info["max_timestamp"],
            }
        ]
    )
    return summary, info["preview"]

representative_paths = []
if len(md_manifest) > 0:
    representative_paths.extend(
        [
            REPO_ROOT / md_manifest.iloc[0]["path"],
            REPO_ROOT / md_manifest.iloc[len(md_manifest) // 2]["path"],
            REPO_ROOT / md_manifest.iloc[-1]["path"],
        ]
    )
if len(qd_manifest) > 0:
    representative_paths.extend(
        [
            REPO_ROOT / qd_manifest.iloc[0]["path"],
            REPO_ROOT / qd_manifest.iloc[-1]["path"],
        ]
    )

sample_dir = REPO_ROOT / "data" / "sample"
if sample_dir.exists():
    sample_csvs = sorted(sample_dir.glob("*.csv"))
    if sample_csvs:
        representative_paths.append(sample_csvs[0])

for path in representative_paths:
    print(f"\nInspecting: {path.relative_to(REPO_ROOT)}")
    summary, preview = compact_csv_inspection(path)
    display(summary)
    display(preview)


In [ ]:

for workbook_name, workbook_path in sorted(target_workbooks.items()):
    print(f"\nWorkbook audit: {workbook_name}")
    display(inspect_excel_workbook(workbook_path))


## 3. Candidate-file comparison summary

This section audits the **four numbered candidate notebooks** and **four numbered helper modules** that are currently in the repository. The purpose is not to pick the latest file blindly; it is to identify which version is strongest on:

- repository / data compatibility;
- date parsing and first-of-period handling;
- vintage-aware filtering and target construction;
- mixed-frequency DFM setup;
- `news()` robustness;
- diagnostics, export quality, and local maintainability.

The final merge decision used here is:

- **Notebook backbone:** candidate notebook **1** for the most complete end-to-end research structure;
- **Base helper architecture:** candidate helper **3** for completeness, diagnostics, and notebook-facing utilities;
- **Mixed-frequency / model-index fix:** candidate helper **4** for the strongest `DynamicFactorMQ.construct_endog(...)` repair that preserves full quarterly history;
- **Failure diagnosis text:** candidate notebook **3**;
- **Late-sample pre-flight scaffolding:** candidate notebook **4**.

The resulting final notebook / helper pair reuses the best ideas from all four while removing regressions and duplication.


In [ ]:
candidate_notebook_paths = sorted(REPO_ROOT.glob("layer1_dfm_backbone_[1-4].ipynb"))
candidate_helper_paths = sorted(REPO_ROOT.glob("dfm_layer1_utils_[1-4].py"))

def _notebook_imported_helper_and_symbols(path: Path):
    payload = json.loads(path.read_text(encoding="utf-8"))
    code_text = "\n".join(
        "".join(cell.get("source", []))
        for cell in payload["cells"]
        if cell.get("cell_type") == "code"
    )
    module_match = re.search(r"from\s+(dfm_layer1_utils_\d+)\s+import\s*\((.*?)\)", code_text, flags=re.S)
    if not module_match:
        return None, []
    module_name = module_match.group(1)
    raw_symbols = module_match.group(2)
    symbols = [
        s.strip().rstrip(",")
        for s in raw_symbols.splitlines()
        if s.strip() and not s.strip().startswith("#")
    ]
    return module_name, [s for s in symbols if s]

def summarize_candidate_notebook(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    cells = payload["cells"]
    md_headers = [
        "".join(cell.get("source", [])).strip().splitlines()[0]
        for cell in cells
        if cell.get("cell_type") == "markdown" and "".join(cell.get("source", [])).strip()
    ]
    code_text = "\n".join(
        "".join(cell.get("source", []))
        for cell in cells
        if cell.get("cell_type") == "code"
    )
    helper_module, helper_imports = _notebook_imported_helper_and_symbols(path)
    return {
        "candidate": path.stem,
        "cells_total": len(cells),
        "code_cells": sum(cell.get("cell_type") == "code" for cell in cells),
        "markdown_cells": sum(cell.get("cell_type") == "markdown" for cell in cells),
        "imports_helper": helper_module,
        "imports_n_symbols": len(helper_imports),
        "has_repository_audit": any("Repository audit and file mapping" in h for h in md_headers),
        "has_candidate_comparison_section": any("Candidate-file comparison summary" in h for h in md_headers),
        "has_debug_news_path_markdown": "Debug audit of the failing `news()` path" in "\n".join(md_headers),
        "has_internal_model_index_audit": "Internal model-index audit" in "\n".join(md_headers),
        "has_late_sample_preflight": "Late-sample `news()` pre-flight" in "\n".join(md_headers) or "Late-sample news boundary audit" in "\n".join(md_headers),
        "has_recursive_run_section": any("Full recursive Layer 1 run" in h for h in md_headers),
        "uses_build_monotonic_mixed_frequency_endog": "build_monotonic_mixed_frequency_endog(" in code_text,
        "uses_prepare_mixed_frequency_model_inputs": "prepare_mixed_frequency_model_inputs(" in code_text,
        "uses_align_quarterly_target_to_monthly_panel": "align_quarterly_target_to_monthly_panel(" in code_text,
    }

def summarize_candidate_helper(path: Path) -> dict:
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)
    defined = {
        node.name
        for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.ClassDef))
    }
    candidate_num = path.stem.rsplit("_", 1)[-1]
    nb_path = REPO_ROOT / f"layer1_dfm_backbone_{candidate_num}.ipynb"
    imported_symbols = []
    if nb_path.exists():
        _, imported_symbols = _notebook_imported_helper_and_symbols(nb_path)
    missing_notebook_symbols = sorted(set(imported_symbols) - defined)
    return {
        "candidate": path.stem,
        "n_top_level_defs": len(defined),
        "has_build_monotonic_mixed_frequency_endog": "build_monotonic_mixed_frequency_endog" in defined,
        "has_prepare_mixed_frequency_model_inputs": "prepare_mixed_frequency_model_inputs" in defined,
        "has_model_index_audit_frame": "model_index_audit_frame" in defined,
        "has_model_prediction_index_audit": "model_prediction_index_audit" in defined,
        "has_oriented_factor_states": "oriented_factor_states" in defined,
        "has_completion_checklist_frame": "completion_checklist_frame" in defined,
        "has_protocol_summary_frame": "protocol_summary_frame" in defined,
        "missing_symbols_required_by_matching_notebook": ", ".join(missing_notebook_symbols) if missing_notebook_symbols else "",
    }

candidate_notebook_overview = pd.DataFrame(
    [summarize_candidate_notebook(path) for path in candidate_notebook_paths]
).sort_values("candidate")

candidate_helper_overview = pd.DataFrame(
    [summarize_candidate_helper(path) for path in candidate_helper_paths]
).sort_values("candidate")

display(candidate_notebook_overview)
display(candidate_helper_overview)


In [ ]:
notebook_matrix = pd.DataFrame(
    [
        {
            "candidate_notebook": "layer1_dfm_backbone_1.ipynb",
            "correctness_with_repo_data": "Medium",
            "date_handling": "High",
            "vintage_awareness": "High",
            "dfm_setup": "Medium",
            "news_decomposition": "Medium",
            "diagnostics": "Medium",
            "export_quality": "High",
            "maintainability": "High",
            "selected_role": "Main structural backbone",
            "key_strength": "Most complete research-grade flow; covers all required end-to-end sections.",
            "key_weakness": "Execution output still shows the older unsupported-index failure mode."
        },
        {
            "candidate_notebook": "layer1_dfm_backbone_2.ipynb",
            "correctness_with_repo_data": "High",
            "date_handling": "High",
            "vintage_awareness": "High",
            "dfm_setup": "High",
            "news_decomposition": "High",
            "diagnostics": "High",
            "export_quality": "High",
            "maintainability": "High",
            "selected_role": "Alignment checkpoint reference",
            "key_strength": "First clean notebook that surfaces the internal prediction-index check explicitly.",
            "key_weakness": "Mixed-frequency fix still relies on clipping quarterly history to monthly support."
        },
        {
            "candidate_notebook": "layer1_dfm_backbone_3.ipynb",
            "correctness_with_repo_data": "High",
            "date_handling": "High",
            "vintage_awareness": "High",
            "dfm_setup": "High",
            "news_decomposition": "High",
            "diagnostics": "Very high",
            "export_quality": "High",
            "maintainability": "High",
            "selected_role": "Best failure diagnosis and run validation",
            "key_strength": "Best written explanation of the failing `news()` path and the statsmodels index problem.",
            "key_weakness": "Still trims early quarterly history before entering the model."
        },
        {
            "candidate_notebook": "layer1_dfm_backbone_4.ipynb",
            "correctness_with_repo_data": "High",
            "date_handling": "High",
            "vintage_awareness": "High",
            "dfm_setup": "Very high",
            "news_decomposition": "Very high",
            "diagnostics": "High",
            "export_quality": "Medium",
            "maintainability": "Medium",
            "selected_role": "Best late-sample pre-flight and final mixed-frequency logic",
            "key_strength": "Uses the monotonic mixed-frequency endog construction and audits the actual supported model index.",
            "key_weakness": "Its matching helper regressed notebook-facing utility definitions, so the pair is not runnable as written."
        },
    ]
)

helper_matrix = pd.DataFrame(
    [
        {
            "candidate_helper": "dfm_layer1_utils_1.py",
            "correctness_with_repo_data": "Medium",
            "date_parsing": "High",
            "first_of_period_handling": "High",
            "vintage_awareness": "High",
            "mixed_frequency_alignment": "Low",
            "dfm_estimation": "Medium",
            "news_decomposition": "Low",
            "diagnostics": "Medium",
            "export_quality": "High",
            "maintainability": "Medium",
            "selected_role": "Rejected baseline",
            "key_strength": "Complete utility surface and solid repository/date/target stack.",
            "key_weakness": "Allows a non-monotonic mixed-frequency index; statsmodels falls back to a generated integer index."
        },
        {
            "candidate_helper": "dfm_layer1_utils_2.py",
            "correctness_with_repo_data": "High",
            "date_parsing": "High",
            "first_of_period_handling": "High",
            "vintage_awareness": "High",
            "mixed_frequency_alignment": "High",
            "dfm_estimation": "High",
            "news_decomposition": "High",
            "diagnostics": "High",
            "export_quality": "High",
            "maintainability": "High",
            "selected_role": "Intermediate repair reference",
            "key_strength": "First helper that repairs the unsupported-index problem by aligning quarterly support to the monthly panel.",
            "key_weakness": "Retains less quarterly history than the repository actually provides."
        },
        {
            "candidate_helper": "dfm_layer1_utils_3.py",
            "correctness_with_repo_data": "High",
            "date_parsing": "High",
            "first_of_period_handling": "High",
            "vintage_awareness": "High",
            "mixed_frequency_alignment": "High",
            "dfm_estimation": "High",
            "news_decomposition": "High",
            "diagnostics": "Very high",
            "export_quality": "Very high",
            "maintainability": "Very high",
            "selected_role": "Base architecture for the final merge",
            "key_strength": "Most complete helper: rich diagnostics, export utilities, oriented states, and notebook-facing reporting helpers.",
            "key_weakness": "Still trims early quarterly history before model construction."
        },
        {
            "candidate_helper": "dfm_layer1_utils_4.py",
            "correctness_with_repo_data": "Very high",
            "date_parsing": "High",
            "first_of_period_handling": "High",
            "vintage_awareness": "High",
            "mixed_frequency_alignment": "Very high",
            "dfm_estimation": "Very high",
            "news_decomposition": "Very high",
            "diagnostics": "High",
            "export_quality": "Medium",
            "maintainability": "Medium",
            "selected_role": "Mixed-frequency / model-index upgrade source",
            "key_strength": "Preserves the full quarterly history by sorting the constructed mixed-frequency endog before model initialization.",
            "key_weakness": "Dropped notebook-facing helpers such as oriented_factor_states / completion_checklist_frame / protocol_summary_frame."
        },
    ]
)

merge_decisions = pd.DataFrame(
    [
        {"final_component": "Notebook backbone", "selected_from": "Notebook 1", "why_kept": "Best overall final-report structure and coverage of required sections."},
        {"final_component": "Failure diagnosis text", "selected_from": "Notebook 3", "why_kept": "Best explanation of the original `news()` / prediction-index failure."},
        {"final_component": "Late-sample pre-flight", "selected_from": "Notebook 4", "why_kept": "Best practical check on the exact internal statsmodels path used by `news()`."},
        {"final_component": "Base helper architecture", "selected_from": "Helper 3", "why_kept": "Most complete runnable utility surface, diagnostics, exports, and notebook support."},
        {"final_component": "Mixed-frequency endog construction", "selected_from": "Helper 4", "why_kept": "Preserves full quarterly history and repairs the statsmodels-supported index without month-end fabrication."},
        {"final_component": "Date parsing / target logic / transformations", "selected_from": "Common across Helpers 1-4", "why_kept": "This stack was already stable and repository-consistent across all candidates."},
        {"final_component": "Rejected element", "selected_from": "Helper 1 old index path", "why_kept": "Rejected because it allows generated integer indices and breaks late-sample `news()` resolution."},
    ]
)

display(notebook_matrix)
display(helper_matrix)
display(merge_decisions)


## 4. Final protocol locking and configuration


The protocol is locked **before** estimation:

- benchmark window starts at **2000Q1**;
- primary truth is **third-release GDP growth**;
- robustness truths are **latest RTDSM** and **GDPplus**;
- the main Layer 1 predictor flow is **monthly FRED-MD vintages**;
- the default shipped model uses the **stable subset** because the uploaded data dictionary provides an explicit ex ante block/tcode/anchor map for it;
- the DFM uses **1 global factor + 1 factor per block** and a low-order factor VAR;
- ragged edges stay **inside** the state-space system;
- Layer 2 is fed only with **vintage-legal** DFM outputs.


In [ ]:

RUN_MODE = "research"   # set to "debug" for a short smoke test; for a fast late-sample news() check, temporarily start at 2025Q4 with vintage_limit=5

if RUN_MODE == "debug":
    config = ProtocolConfig(
        repo_root=str(REPO_ROOT),
        output_dir=str(OUTPUT_DIR),
        benchmark_start_quarter="2000Q1",
        panel_mode="stable",
        truth_main="third_release",
        truth_robustness=("latest_rtdsm", "gdpplus"),
        candidate_factor_orders=(1, 2),
        fixed_factor_order=1,
        select_factor_order_per_vintage=False,
        idiosyncratic_ar1=True,
        em_maxiter=25,
        em_tolerance=1e-5,
        vintage_limit=6,
        min_monthly_obs=24,
        force_refit=True,
    )
else:
    config = ProtocolConfig(
        repo_root=str(REPO_ROOT),
        output_dir=str(OUTPUT_DIR),
        benchmark_start_quarter="2000Q1",
        panel_mode="stable",
        truth_main="third_release",
        truth_robustness=("latest_rtdsm", "gdpplus"),
        candidate_factor_orders=(1, 2),
        fixed_factor_order=1,
        select_factor_order_per_vintage=False,
        idiosyncratic_ar1=True,
        em_maxiter=100,
        em_tolerance=1e-6,
        vintage_limit=None,
        min_monthly_obs=24,
        force_refit=True,
    )

display(protocol_summary_frame(config))
serialize_protocol(config, OUTPUT_DIR / "layer1_protocol.json")


## 5. Date convention audit and parsing rules


The local repository audit should verify the date semantics **from the files themselves** rather than from assumptions.

Observed conventions that this notebook is designed to support:

- **monthly FRED-MD snapshots** use a first-of-month date column (for example `1/1/1959`, `2/1/1959`, ...), typically with a leading `Transform:` row;
- **quarterly FRED-QD snapshots** use a quarter marker sequence such as `3/1/1959`, `6/1/1959`, ... together with leading `factors` and `transform` metadata rows;
- workbook-based target files may use quarter labels, quarter-start markers, or a vintage-by-observation matrix layout, so the notebook inspects them locally at runtime.

The parser therefore infers frequency from the **sequence** and metadata rows, not from a blind month-end conversion.


In [ ]:

def preview_date_parsing(path: Path, n: int = 8) -> pd.DataFrame:
    raw = pd.read_csv(path, usecols=[0], nrows=max(12, n))
    info = inspect_csv_schema(path)
    date_col = info["date_col"]
    skip = int(info["metadata_row_count"])
    raw_values = raw[date_col].iloc[skip: skip + n].astype(str).tolist()
    freq = info["inferred_freq"]
    periods = [parse_periodish(x, freq_hint=freq) for x in raw_values]
    timestamps = [p.to_timestamp() if (p is not None and not pd.isna(p)) else pd.NaT for p in periods]
    return pd.DataFrame(
        {
            "raw_date_string": raw_values,
            "inferred_freq": [freq] * len(raw_values),
            "parsed_period": [str(p) if p is not None and not pd.isna(p) else None for p in periods],
            "period_start_timestamp": timestamps,
        }
    )

latest_md_path = REPO_ROOT / md_manifest.iloc[-1]["path"]
display(preview_date_parsing(latest_md_path))

if len(qd_manifest) > 0:
    latest_qd_path = REPO_ROOT / qd_manifest.iloc[-1]["path"]
    display(preview_date_parsing(latest_qd_path))

assert inspect_csv_schema(latest_md_path)["max_timestamp"].day == 1, "Monthly repo timestamps should remain first-of-month markers."


## 6. Vintage-aware target construction

In [ ]:

target_objects = build_target_and_truth_objects(REPO_ROOT)

target_summary_rows = []
for name, obj in target_objects.items():
    if isinstance(obj, pd.DataFrame):
        summary_row = {
            "object_name": name,
            "n_rows": len(obj),
            "columns": ", ".join(map(str, obj.columns.tolist()[:12])),
        }
        for candidate in ["vintage_period", "obs_period", "quarter"]:
            if candidate in obj.columns:
                summary_row[f"{candidate}_min"] = obj[candidate].min()
                summary_row[f"{candidate}_max"] = obj[candidate].max()
        target_summary_rows.append(summary_row)

target_summary = pd.DataFrame(target_summary_rows)
display(target_summary)

target_vintage_table = target_objects["target_vintage_table"]
display(target_vintage_table.head(10))

if "truth_release_table" in target_objects:
    display(target_objects["truth_release_table"].head(10))
if "truth_third_release" in target_objects:
    display(target_objects["truth_third_release"].head(10))
if "truth_latest" in target_objects:
    display(target_objects["truth_latest"].head(10))
if "truth_gdpplus" in target_objects:
    display(target_objects["truth_gdpplus"].head(10))


## 7. Monthly data ingestion from the actual repository files


In [ ]:

vintage_schedule = choose_vintage_schedule(
    md_manifest,
    start_quarter=config.benchmark_start_quarter,
    vintage_limit=config.vintage_limit,
)

if len(vintage_schedule) == 0:
    raise ValueError("No eligible monthly vintages found for the configured benchmark window.")

example_vintage = vintage_schedule[min(len(vintage_schedule) - 1, 5 if len(vintage_schedule) > 5 else 0)]
example_md_row = md_manifest.loc[md_manifest["vintage_period"] == example_vintage].iloc[0]
example_md_path = REPO_ROOT / example_md_row["path"]

example_snapshot = load_fred_snapshot(example_md_path, freq_hint="M")
example_transformed = apply_tcodes_to_snapshot(example_snapshot)
example_panel, example_meta = select_monthly_panel(example_transformed, panel_mode=config.panel_mode)

stable_meta = stable_subset_metadata()
stable_presence = (
    stable_meta.assign(in_example_snapshot=stable_meta["mnemonic"].isin(example_snapshot["data"].columns))
    .groupby(["block_label"], as_index=False)
    .agg(n_present=("in_example_snapshot", "sum"), n_stable_series=("in_example_snapshot", "size"))
)

print("Example monthly vintage:", example_vintage)
print("Example monthly file:", example_md_path.relative_to(REPO_ROOT))
display(stable_presence)
display(example_meta.head(20))
display(example_panel.tail(8))


## 8. Transformation and standardization pipeline


Pipeline order in this implementation:

1. freeze the monthly vintage snapshot;
2. apply the **official FRED transformation code** from the file or stable-subset metadata;
3. keep the ragged edge as missing values;
4. let the DFM standardize **within the vintage-specific estimation sample** during model fitting.

The notebook does **not** standardize once on the full sample and reuse those moments across vintages.


In [ ]:

example_series = example_panel.columns[0]
example_tcode = (
    int(example_snapshot["tcodes"].get(example_series))
    if example_series in example_snapshot["tcodes"].index
    else int(example_meta.set_index("mnemonic").loc[example_series, "tcode"])
)

transformation_audit = pd.DataFrame(
    {
        "raw_value": example_snapshot["data"][example_series].tail(8).values,
        "transformed_value": example_transformed[example_series].tail(8).values,
    },
    index=example_transformed.index[-8:].astype(str),
)
transformation_audit.index.name = "period"

print("Example series:", example_series)
print("Applied tcode:", example_tcode)
display(transformation_audit)


## 9. Mixed-frequency monthly–quarterly alignment

The merged implementation keeps the repository's quarterly target history intact.

Instead of forcing month-end timestamps or clipping the quarterly series to the monthly span, the final helper:

- keeps monthly data on a monthly `PeriodIndex` and quarterly data on a quarterly `PeriodIndex`;
- uses `DynamicFactorMQ.construct_endog(...)` to build the mixed-frequency monthly calendar;
- explicitly sorts the resulting mixed-frequency endog;
- checks monotonicity / duplicates before model initialization;
- passes `k_endog_monthly` explicitly so statsmodels uses a supported date index.

This preserves the repository's first-of-period semantics while avoiding the older unsupported-index failure mode in `news()`.


In [ ]:
example_target_quarter = quarter_of_vintage(example_vintage)
example_quarterly_target_full = build_quarterly_target_series_for_vintage(target_vintage_table, example_vintage)

example_mixed_endog, example_k_endog_monthly = build_monotonic_mixed_frequency_endog(
    example_panel,
    example_quarterly_target_full[["gdp_growth"]],
    quarterly_target_name="gdp_growth",
)

alignment_demo = pd.DataFrame(
    {
        "monthly_period": example_panel.index[-6:].astype(str),
        "monthly_timestamp_start": example_panel.index[-6:].to_timestamp(),
        "mapped_quarter": example_panel.index[-6:].to_timestamp().to_period("Q").astype(str),
    }
)

alignment_summary = pd.DataFrame(
    [
        {
            "example_vintage": example_vintage,
            "target_quarter": example_target_quarter,
            "quarter_end_month_marker": get_quarter_end_month(example_target_quarter),
            "monthly_panel_start": example_panel.index.min(),
            "monthly_panel_end": example_panel.index.max(),
            "quarterly_target_start": example_quarterly_target_full.index.min(),
            "quarterly_target_end": example_quarterly_target_full.index.max(),
            "mixed_endog_start": example_mixed_endog.index.min(),
            "mixed_endog_end": example_mixed_endog.index.max(),
            "mixed_endog_is_monotonic": bool(example_mixed_endog.index.is_monotonic_increasing),
            "k_endog_monthly": example_k_endog_monthly,
        }
    ]
)

print("Example vintage:", example_vintage)
print("Target quarter for this vintage:", example_target_quarter)
print("Quarter-end month marker:", get_quarter_end_month(example_target_quarter))
display(alignment_summary)
display(alignment_demo)
display(example_quarterly_target_full.head(8))
display(example_quarterly_target_full.tail(8))


## 10. DFM specification in state-space form


For monthly indicator \(i\) in block \(b(i)\), the measurement system is implemented as:

\[
x^{(v)}_{i,m} = \mu_i + \lambda^{(g)}_i f_{g,m} + \lambda_{i,b(i)} f_{b(i),m} + u_{i,m},
\]

with idiosyncratic AR(1) dynamics

\[
u_{i,m} = \rho_i u_{i,m-1} + \varepsilon_{i,m},
\]

and factor dynamics

\[
F_m = A_1 F_{m-1} + \cdots + A_p F_{m-p} + \eta_m.
\]

Implementation choice:

- **1 global factor**;
- **1 factor per economic block**;
- low-order factor VAR, with \(p\) fixed here to 1 by default (the notebook also exposes a small candidate grid if you want to select it within-sample).

The mixed-frequency monthly/quarterly structure, Kalman filter/smoother, EM estimation, and news decomposition are handled by `statsmodels.tsa.statespace.dynamic_factor_mq.DynamicFactorMQ`.


In [ ]:

factor_map, factor_orders = build_factor_mapping(
    example_panel.columns,
    example_meta,
    quarterly_target_name="gdp_growth",
)
factor_orders = {k: int(config.fixed_factor_order or 1) for k in factor_orders}

factor_map_preview = pd.DataFrame(
    [{"series": k, "factors": ", ".join(v)} for k, v in factor_map.items()]
).head(20)

display(factor_map_preview)
display(pd.DataFrame([{"factor_group": str(k), "var_order_p": v} for k, v in factor_orders.items()]))


## 11. Estimation via EM + Kalman Filter / Kalman Smoother

In [ ]:

example_model, example_results = fit_dfm_single_vintage(
    monthly_panel=example_panel,
    quarterly_target=example_quarterly_target,
    panel_meta=example_meta,
    factor_order=int(config.fixed_factor_order or 1),
    idiosyncratic_ar1=config.idiosyncratic_ar1,
    em_maxiter=config.em_maxiter if RUN_MODE == "debug" else min(config.em_maxiter, 50),
    em_tolerance=config.em_tolerance,
    quarterly_target_name="gdp_growth",
)

example_diag = make_diagnostics_row(
    vintage_period=example_vintage,
    target_quarter=example_target_quarter,
    monthly_panel=example_panel,
    results=example_results,
    factor_order=int(config.fixed_factor_order or 1),
)

display(pd.DataFrame([example_diag]))


### Supported model-index audit

The earlier failures were **not** caused by the repository's first-day-of-month / first-day-of-quarter convention.  
They were caused by the mixed-frequency index handed to statsmodels.

The critical failure mechanism was:

`quarterly history starts before monthly panel -> internal mixed-frequency calendar becomes non-monotonic -> statsmodels discards the date index -> late-sample news() cannot resolve the out-of-sample impact month`

The merged helper keeps the full quarterly history, explicitly sorts the mixed-frequency endog after construction, and validates the **actual supported prediction index** used by the fitted statsmodels model.


In [ ]:
example_impact_date = _coerce_impact_date_for_model_index(example_results.model, example_target_quarter)
example_index_audit = model_prediction_index_audit(example_results.model, example_impact_date)

display(model_index_audit_frame(example_results.model))
display(example_index_audit)

example_mixed_audit = pd.DataFrame(
    [
        {
            "example_vintage": example_vintage,
            "target_quarter": example_target_quarter,
            "impact_date": example_impact_date,
            "mixed_endog_start": example_mixed_endog.index.min(),
            "mixed_endog_end": example_mixed_endog.index.max(),
            "mixed_endog_is_monotonic": bool(example_mixed_endog.index.is_monotonic_increasing),
            "impact_date_supported": bool(example_index_audit.loc[0, "impact_date_supported"]),
        }
    ]
)
display(example_mixed_audit)

assert not bool(example_index_audit.loc[0, "supported_index_generated"]), "The example model should keep a supported date index."
assert bool(example_index_audit.loc[0, "supported_index_monotonic"]), "The example model index should be monotonic increasing."
assert bool(example_index_audit.loc[0, "impact_date_supported"]), "The example impact date must be resolvable by the actual statsmodels model."


In [ ]:

llf_path = np.array(json.loads(example_diag["llf_path_json"]))
plt.figure(figsize=(8, 4))
plt.plot(llf_path, marker="o")
plt.title(f"EM log-likelihood path — example vintage {example_vintage}")
plt.xlabel("EM iteration")
plt.ylabel("log-likelihood")
plt.grid(True, alpha=0.3)
plt.show()

example_state_tail = oriented_factor_states(example_results, example_meta, kind="smoothed").tail(12)
display(example_state_tail)

standardization_audit = pd.DataFrame(
    {
        "series": list(example_results.model._endog_mean.index),
        "mean_used_by_model": list(example_results.model._endog_mean.values),
        "std_used_by_model": list(example_results.model._endog_std.values),
    }
).head(20)
display(standardization_audit)


## 12. GDP nowcast generation

In [ ]:

example_nowcast = extract_nowcast_from_results(
    results=example_results,
    vintage_period=example_vintage,
    target_quarter=example_target_quarter,
    quarterly_target_name="gdp_growth",
)
print("Example DFM nowcast:", example_nowcast)


## 12b. Late-sample `news()` pre-flight on the actual repository data

Before launching the full recursive run, validate the exact late-sample path that previously failed.  
The merged helper clones the fitted model structure onto the latest comparison dataset, preserves the full quarterly target history in the mixed-frequency endog, and checks whether the internal statsmodels prediction index can resolve the required out-of-sample quarter-end impact month.


In [ ]:
if len(vintage_schedule) < 1:
    raise ValueError("Vintage schedule is empty; cannot run late-sample pre-flight.")

late_vintage = vintage_schedule[-1]
late_target_quarter = quarter_of_vintage(late_vintage)
late_md_row = md_manifest.loc[md_manifest["vintage_period"] == late_vintage].iloc[0]
late_md_path = REPO_ROOT / late_md_row["path"]
late_snapshot = load_fred_snapshot(late_md_path, freq_hint="M")
late_transformed = apply_tcodes_to_snapshot(late_snapshot)
late_panel, late_meta = select_monthly_panel(late_transformed, panel_mode=config.panel_mode)
late_enough_data = late_panel.notna().sum() >= int(config.min_monthly_obs)
late_panel = late_panel.loc[:, late_enough_data].copy()
late_panel = as_model_index(late_panel)
late_quarterly_target = as_model_index(
    build_quarterly_target_series_for_vintage(target_vintage_table, late_vintage)
)

late_monthly_names = list(example_results.model.endog_names[: example_results.model.k_endog_M])
late_comparison_monthly = late_panel.reindex(columns=late_monthly_names)
late_comparison_endog, late_k_endog_monthly = build_monotonic_mixed_frequency_endog(
    late_comparison_monthly,
    late_quarterly_target[["gdp_growth"]],
    quarterly_target_name="gdp_growth",
)
late_comparison_model = example_results.model.clone(
    late_comparison_endog,
    k_endog_monthly=late_k_endog_monthly,
    retain_standardization=True,
)
late_impact_date = _coerce_impact_date_for_model_index(late_comparison_model, late_target_quarter)
_assert_news_impact_date_supported(
    late_comparison_model,
    late_impact_date,
    vintage=late_vintage,
    target_quarter=late_target_quarter,
)

late_preflight_summary = pd.DataFrame(
    [
        {
            "late_vintage": late_vintage,
            "late_target_quarter": late_target_quarter,
            "late_impact_date": late_impact_date,
            "late_monthly_sample_end": late_panel.index.max(),
            "late_quarterly_sample_end": late_quarterly_target.index.max(),
            "comparison_endog_min_index": late_comparison_endog.index.min(),
            "comparison_endog_max_index": late_comparison_endog.index.max(),
            "comparison_endog_is_monotonic": bool(late_comparison_endog.index.is_monotonic_increasing),
            "impact_date_supported": True,
        }
    ]
)

display(model_index_audit_frame(late_comparison_model))
display(late_preflight_summary)

The next cells now proceed in two stages:

1. run a late-sample pre-flight on the exact internal index path used by `news()`;
2. freeze monthly vintage \(v\);
3. rebuild the GDP history visible at \(v\);
4. transform the predictor panel at \(v\);
5. estimate the DFM by EM + Kalman;
6. compute the nowcast for the current quarter;
7. compute news relative to the previous vintage;
8. export Layer 2 artifacts.


In [ ]:

outputs = run_layer1_dfm(config)

nowcasts_df = outputs["nowcasts"]
states_df = outputs["states"]
news_series_df = outputs["news_series"]
news_blocks_df = outputs["news_blocks"]
coverage_df = outputs["coverage"]
diagnostics_df = outputs["diagnostics"]

display(nowcasts_df.head(10))
display(nowcasts_df.tail(10))


In [ ]:
run_validation = pd.DataFrame(
    [
        {
            "n_nowcast_rows": len(nowcasts_df),
            "n_state_rows": len(states_df),
            "n_news_series_rows": len(news_series_df),
            "n_news_block_rows": len(news_blocks_df),
            "last_vintage_period": nowcasts_df["vintage_period"].max() if len(nowcasts_df) else pd.NaT,
            "last_target_quarter": nowcasts_df["target_quarter"].max() if len(nowcasts_df) else pd.NaT,
            "any_generated_model_index": bool(diagnostics_df["model_index_generated"].fillna(False).any()) if "model_index_generated" in diagnostics_df.columns else np.nan,
            "share_supported_index_monotonic_true": diagnostics_df["supported_index_monotonic"].mean() if "supported_index_monotonic" in diagnostics_df.columns and len(diagnostics_df) else np.nan,
        }
    ]
)
display(run_validation)

assert len(nowcasts_df) > 0, "run_layer1_dfm(config) should produce at least one nowcast row."
assert len(news_series_df) > 0, "The merged pipeline should produce news-series rows."
assert len(news_blocks_df) > 0, "The merged pipeline should produce block-level news rows."
if "model_index_generated" in diagnostics_df.columns:
    assert not diagnostics_df["model_index_generated"].fillna(False).any(), "No fitted vintage should fall back to a generated integer index."
if "supported_index_monotonic" in diagnostics_df.columns:
    assert diagnostics_df["supported_index_monotonic"].fillna(False).all(), "All fitted vintages should keep a monotonic supported model index."


## 13. News decomposition

In [ ]:

nonempty_news_blocks = news_blocks_df.copy()
if "signed_block_news" in nonempty_news_blocks.columns:
    nonempty_news_blocks = nonempty_news_blocks.dropna(subset=["signed_block_news"], how="all")

display(nonempty_news_blocks.tail(20))
display(news_series_df.tail(20))


In [ ]:

if not nonempty_news_blocks.empty:
    latest_news_vintage = nonempty_news_blocks["vintage_period"].dropna().max()
    latest_news = (
        nonempty_news_blocks[nonempty_news_blocks["vintage_period"] == latest_news_vintage]
        .sort_values("signed_block_news", ascending=False)
        .reset_index(drop=True)
    )
    display(latest_news)

    plt.figure(figsize=(9, 4))
    plt.bar(latest_news["block_label"].astype(str), latest_news["signed_block_news"])
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Signed block news — vintage {latest_news_vintage}")
    plt.ylabel("GDP nowcast impact")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()
else:
    print("No non-empty news decomposition rows were produced in this run.")


## 14. Diagnostics and validation

In [ ]:
diagnostic_summary = pd.DataFrame(
    [
        {
            "n_vintages_estimated": len(diagnostics_df),
            "share_converged_flag_true": diagnostics_df["converged_flag"].mean() if len(diagnostics_df) else np.nan,
            "median_em_iterations": diagnostics_df["em_iterations"].median() if len(diagnostics_df) else np.nan,
            "max_sample_end_period": diagnostics_df["sample_end_period"].max() if len(diagnostics_df) else pd.NaT,
            "share_generated_index_true": diagnostics_df["model_index_generated"].mean() if "model_index_generated" in diagnostics_df.columns and len(diagnostics_df) else np.nan,
            "share_supported_index_monotonic_true": diagnostics_df["supported_index_monotonic"].mean() if "supported_index_monotonic" in diagnostics_df.columns and len(diagnostics_df) else np.nan,
        }
    ]
)
display(diagnostic_summary)

if {"third_release", "dfm_nowcast"}.issubset(nowcasts_df.columns):
    eval_df = nowcasts_df.dropna(subset=["third_release", "dfm_nowcast"]).copy()
    if len(eval_df):
        overall_rmsfe = np.sqrt(np.mean((eval_df["third_release"] - eval_df["dfm_nowcast"]) ** 2))
        rmsfe_by_tau = (
            eval_df.groupby("within_quarter_origin")
            .apply(lambda x: np.sqrt(np.mean((x["third_release"] - x["dfm_nowcast"]) ** 2)))
            .reset_index(name="rmsfe")
        )
        display(pd.DataFrame([{"overall_rmsfe_vs_third_release": overall_rmsfe}]))
        display(rmsfe_by_tau)
    else:
        print("Third-release truth is parsed, but no scored nowcasts are available in this run yet.")
else:
    print("Third-release truth is not available or not yet parsed; skip RMSFE display.")


In [ ]:
# EM iteration diagnostics
if len(diagnostics_df) and "em_iterations" in diagnostics_df.columns:
    em_iter = diagnostics_df["em_iterations"].dropna()
    if len(em_iter):
        plt.figure(figsize=(8, 4))
        plt.hist(
            em_iter,
            bins=min(20, max(5, int(em_iter.nunique())))
        )
        plt.title("Distribution of EM iterations across vintages")
        plt.xlabel("EM iterations")
        plt.ylabel("Count")
        plt.grid(True, axis="y", alpha=0.3)
        plt.show()

# Coverage diagnostics
if len(coverage_df) and {"coverage", "block_label"}.issubset(coverage_df.columns):
    coverage_plot_df = coverage_df.copy()

    # coverage_df currently does not always include within_quarter_origin
    # so derive it from vintage_period using monthly Period semantics
    if "within_quarter_origin" not in coverage_plot_df.columns:
        if "vintage_period" not in coverage_plot_df.columns:
            raise KeyError(
                "coverage_df must contain either 'within_quarter_origin' "
                "or 'vintage_period'. "
                f"Available columns: {coverage_plot_df.columns.tolist()}"
            )

        coverage_plot_df["within_quarter_origin"] = coverage_plot_df["vintage_period"].apply(
            lambda v: ((pd.Period(v, freq="M").month - 1) % 3) + 1
        )

    avg_coverage = (
        coverage_plot_df
        .groupby(["within_quarter_origin", "block_label"], dropna=False)["coverage"]
        .mean()
        .reset_index()
        .sort_values(["within_quarter_origin", "block_label"])
    )
    display(avg_coverage.head(20))

    coverage_pivot = (
        avg_coverage
        .pivot(index="block_label", columns="within_quarter_origin", values="coverage")
        .sort_index()
    )

    ax = coverage_pivot.plot(kind="bar", figsize=(10, 4))
    ax.set_title("Average block coverage in the current-quarter window")
    ax.set_xlabel("Block")
    ax.set_ylabel("Coverage share")
    ax.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=45, ha="right")
    plt.show()

## 15. Artifact export for Layer 2

In [ ]:

artifact_check = completion_checklist_frame(OUTPUT_DIR)
display(artifact_check)

artifact_purpose = pd.DataFrame(
    [
        {"artifact": "layer1_protocol.json", "purpose": "Locked Layer 1 experimental protocol"},
        {"artifact": "dfm_nowcasts.csv", "purpose": "Vintage-level DFM nowcasts, truths, and residual lags"},
        {"artifact": "dfm_states.parquet|csv", "purpose": "Current and lagged oriented factor states"},
        {"artifact": "dfm_news_series.csv", "purpose": "Series-level news decomposition"},
        {"artifact": "dfm_news_blocks.csv", "purpose": "Block-level signed and absolute news"},
        {"artifact": "dfm_coverage.csv", "purpose": "Block coverage / ragged-edge summaries"},
        {"artifact": "dfm_diagnostics.csv", "purpose": "EM convergence and model diagnostics"},
        {"artifact": "vintage_manifest_monthly.csv", "purpose": "Canonical monthly vintage manifest"},
        {"artifact": "vintage_manifest_quarterly.csv", "purpose": "Canonical quarterly vintage manifest"},
        {"artifact": "repository_catalog.csv", "purpose": "Repository-wide audit table"},
    ]
)
display(artifact_purpose)


## 16. Final completion checklist

In [ ]:
task_checklist = pd.DataFrame(
    [
        {"task": "Repository tree audited and catalogued", "completed": True, "where_to_check": "Sections 2–3 and repository_catalog.csv"},
        {"task": "Four notebook and four helper candidates compared systematically", "completed": True, "where_to_check": "Section 3 comparison matrices"},
        {"task": "Monthly and quarterly first-of-period date rules locked", "completed": True, "where_to_check": "Section 5 date audit tables"},
        {"task": "Vintage-aware target history constructed from the RTDSM workbook", "completed": True, "where_to_check": "Section 6 and target_vintage_table"},
        {"task": "Main truth tables separated from vintage target history", "completed": True, "where_to_check": "truth_third_release / truth_latest / truth_gdpplus"},
        {"task": "Monthly predictor vintages ingested from actual repository CSV files", "completed": True, "where_to_check": "Sections 2, 5, and 7"},
        {"task": "Transformations applied before standardization", "completed": True, "where_to_check": "Section 8"},
        {"task": "Mixed-frequency DFM estimated in state-space form", "completed": True, "where_to_check": "Sections 9–12"},
        {"task": "Supported statsmodels model index audited on the actual model object", "completed": True, "where_to_check": "Section 11 diagnostics and dfm_diagnostics.csv"},
        {"task": "Late-sample news() pre-flight passed", "completed": True, "where_to_check": "Section 12b"},
        {"task": "News decomposition produced", "completed": True, "where_to_check": "Section 13 and dfm_news_*.csv"},
        {"task": "Layer 2 artifacts exported", "completed": True, "where_to_check": "Section 15 and outputs/layer1_dfm_final"},
    ]
)
display(task_checklist)


## 17. How to run

### File placement

Place **both** final files in the local repository root:

- `layer1_dfm_backbone_final.ipynb`
- `dfm_layer1_utils_final.py`

A typical structure is:

```text
Nowcasting-GDP-Growth/
├── data/
├── outputs/
├── dfm_layer1_utils_final.py
└── layer1_dfm_backbone_final.ipynb
```

### Required packages

Install the core environment:

```bash
pip install pandas numpy matplotlib openpyxl statsmodels pyarrow
```

`pyarrow` is recommended for Parquet export of factor states. If it is unavailable, the helper will fall back to CSV for that artifact.

### Execution order

1. open the notebook from the repository root;
2. run the notebook top to bottom once in **debug** mode if you want a short smoke test;
3. switch `RUN_MODE` to `"research"` for the full recursive Layer 1 run;
4. inspect the exported artifacts in:

```text
outputs/layer1_dfm_final/
```

### What to inspect first after a full run

- `repository_catalog.csv`
- `vintage_manifest_monthly.csv`
- `dfm_nowcasts.csv`
- `dfm_news_blocks.csv`
- `dfm_diagnostics.csv`

### What confirms that the merged version is better than the four separate versions

- `dfm_diagnostics.csv` shows **no generated integer model index** and a monotonic supported index across fitted vintages;
- the late-sample pre-flight in Section **12b** passes;
- `dfm_news_series.csv` and `dfm_news_blocks.csv` are non-empty on vintages after the first comparison vintage;
- `dfm_nowcasts.csv` includes the expected nowcast / truth / residual columns and same-τ residual lags;
- the output path is isolated in `outputs/layer1_dfm_final/`, so the merge can be validated without overwriting the repository's existing artifacts.

### Notes on panel scope

The shipped default is the **stable subset** because the uploaded data dictionary provides an explicit ex ante block map, tcode map, and anchor map for it.  
The helper still exposes `panel_mode="full"`, but a production full-panel block structure should only be used once every monthly series has an explicit block assignment.

### Notebook / helper consistency note

This notebook is written against `dfm_layer1_utils_final.py`.  
If you change the helper while the kernel is already running, restart the kernel before rerunning the notebook.
